# Rocky Mountain anorthosite-granite intrusions ca. 1430 Ma paleomagnetic pole

## Geologic context

The "Rocky Mountain intrusions" pole combines three ~1.4 Ga
anorogenic intrusions of the Colorado-Wyoming province: the Laramie Anorthosite
Complex and the Sherman Granite (Harlan et al., 1994), and the Electra Lake
gabbro (Harlan et al., 1998). These plutons span ca. 1415-1445 Ma. The remanence
is carried by magnetite; the igneous layering of the anorthosite and the
multiple intrusions provide stability tests.

## Pole

This notebook recreates the site-level data for the three intrusions, applies the
igneous-layering ("fold") and reversal tests, and reports the prior-compilation
mean pole (the mean of the three study poles). The repo's contribution
(`data/1430_Rocky/`, built from Harlan et al. 1994 + 1998) supersedes the
Laramie+Sherman-only student submission.

## Age constraints

- Current bracket: nominal age 1430 Ma,  lomagage 1415 Ma, himagage 1445 Ma. 
- Proposed new bracket: nominal age 1430 Ma, lomagage 1415 Ma, himagage 1435 Ma.
- Compilation method note: the Laramie Anorthosite Complex and the Sherman Granite are from the southern Laramie Range of Wyoming and Colorado. Harlan et al., 1994 constructed a thermal history using zircon U-Pb and Ar geochronology from the two units with paleomagnetic evidence that they have overlapping directions. The estimated age of remanence is ca. 1435 Ma for the Laramie Anorthosite Complex and 1415 Ma for the Sherman Granite. The Electra Lake gabbro intrudes 1.7 to 1.6 Ga gneisses and schists of the Needle Mountain in Colorado. The remanence is estimated to be ca. 1435 Ma based on U-Pb zircon age and inferred to have cooled rapidly based on Rb-Sr and Ar geochronology data (Gonzales et al., 1994, Bickford et al., 1969, Harlan and Geissman, 1998). 


## References

### Paleomagnetism
- Harlan, S. S., Snee, L. W., Geissman, J. W., & Brearley, A. J. (1994). Paleomagnetism of the Middle Proterozoic Laramie anorthosite complex and Sherman Granite, southern Laramie Range, Wyoming and Colorado. Journal of Geophysical Research: Solid Earth, 99(B9), 17997–18020. https://doi.org/10.1029/94JB00580
- Harlan, S. S., & Geissman, J. W. (1998). Paleomagnetism of the Middle Proterozoic Electra Lake Gabbro, Needle Mountains, southwestern Colorado. Journal of Geophysical Research: Solid Earth, 103(B7), 15497–15507. https://doi.org/10.1029/98jb01350


### Geochronology
- 

In [ ]:
import pole_tools as pt
import pmagpy.ipmag as ipmag
import pmagpy.pmag as pmag
import matplotlib.pyplot as plt
import pandas as pd

%config InlineBackend.figure_format='retina'
%matplotlib inline

## Site-level data (Harlan et al. 1994 + 1998)

Each intrusion's site means are loaded from `../data/1430_Rocky/sites.txt`, with
both in-situ (`dir_tilt_correction == 0`) and structurally-corrected
(`== 100`) directions; VGPs are computed from the tilt-corrected directions.

In [ ]:
sites_geo, sites_tc = pt.load_magic_sites('../data/1430_Rocky/sites.txt')
sites_tc = ipmag.vgp_calc(sites_tc.copy(), tilt_correction='yes', site_lon='lon',
                          site_lat='lat', dec_tc='dir_dec', inc_tc='dir_inc')
study_lat, study_lon = 40.3, 253.8
print('rock-type codes:', sites_tc['description'].value_counts().to_dict())
print(f'{len(sites_tc)} site means')
sites_tc[['site', 'description', 'lat', 'lon', 'dir_dec', 'dir_inc', 'dir_k', 'vgp_lat', 'vgp_lon']].head(10)

## Plot site locations

There appear to be a few typos in the dataset. The longitude of La2 was reported as 245.511 but was likely meant to be 254.511. The latitude of La5 was reported as 51.780 but was likely meant to be 41.780. 

In [ ]:
pt.plot_site_map(sites_geo)

## Add site VGPs

In [ ]:
sites_geo['vgp_lon'], sites_geo['vgp_lat'], sites_geo['vgp_dp'], sites_geo['vgp_dm'] = pmag.dia_vgp(sites_geo['dir_dec'], sites_geo['dir_inc'], sites_geo['dir_alpha95'], sites_geo['lat'], sites_geo['lon'])
sites_tc['vgp_lon'], sites_tc['vgp_lat'], sites_tc['vgp_dp'], sites_tc['vgp_dm'] = pmag.dia_vgp(sites_tc['dir_dec'], sites_tc['dir_inc'], sites_tc['dir_alpha95'], sites_tc['lat'], sites_tc['lon'])

In [ ]:
sites_combo_df = pd.concat([sites_geo, sites_tc], ignore_index=True, sort=False)

In [ ]:
pmag.magic_write('../data/1430_Rocky/sites.txt', sites_combo_df, 'sites', dataframe=True)

## Sub-poles by intrusion and the combined mean

The Laramie Anorthosite (An / Sy), Sherman Granite (ShGr), and Electra Lake gabbro
(Pgb) sub-poles are computed; the compilation mean is the mean of the study poles.

In [ ]:
groups = {'Laramie Anorthosite': ['An', 'Sy', 'Tr'], 'Sherman Granite': ['ShGr'],
          'Electra Lake gabbro': ['Pgb', 'Pdb']}
vgp_blocks = []
study_poles = []
subs = []
for name, codes in groups.items():
    sub = sites_geo[sites_geo['description'].isin(codes)]
    if len(sub):
        vgp_block, sp = pt.compute_mean_pole(sub, unify_polarity=True, flip=True)
        study_poles.append(sp)
        vgp_blocks.append(vgp_block)
        subs.append(sub)
        pole_plot = pt.plot_vgps_and_pole(vgp_block, sp, figsize=(5,5), central_longitude=200)
        plt.title(name)
        print(f"{name:22s}: {sp['inc']:.1f}/{sp['dec']:.1f} A95 {sp['alpha95']:.1f} N {int(sp['n'])}")

print('\nprior compilation mean Rocky Mountain intrusions (mean of study poles):')
print('  -11.9 N/217.4 E, A95 9.7, N=58; mean direction 41.1/-46.6; age ~1430 Ma')

## Mean of all VGPs

In [ ]:
vgp_block_all, pole_mean = pt.compute_mean_pole(sites_geo, flip=True)
ipmag.print_pole_mean(pole_mean)
pole_plot = pt.plot_vgps_and_pole(vgp_block_all, pole_mean, figsize=(5,5), central_longitude=200)

## Do the VGPs pass a common means test?

In [ ]:
di_la = ipmag.make_di_block(subs[0]['dir_dec'].tolist(), subs[0]['dir_inc'].tolist())
di_sg = ipmag.make_di_block(subs[1]['dir_dec'].tolist(), subs[1]['dir_inc'].tolist())
di_elg = ipmag.make_di_block(subs[2]['dir_dec'].tolist(), subs[2]['dir_inc'].tolist())

ipmag.common_mean_bootstrap(di_la, di_sg)
plt.show()
ipmag.common_mean_bootstrap(di_la, di_elg)
plt.show()
ipmag.common_mean_bootstrap(di_sg, di_elg)
plt.show()

All three pairs fail a common means test.

## VGPs on Equal Area Plots (FIX SO DIRECTIONS!)

In [ ]:
for name, codes in groups.items():
    sub = sites_geo[sites_geo['description'].isin(codes)]
    if len(sub):
        vgp_block, sp = pt.compute_mean_pole(sub, unify_polarity=True, flip=True)
        ipmag.plot_net()
        ipmag.plot_di(di_block=vgp_block, color='blue', marker='o')
        ipmag.plot_di_mean(sp['dec'], sp['inc'], sp['alpha95'], 
                   color='red', marker='s')
        ipmag.plt.title(name)
        plt.show()

In [ ]:
dir_block, dir_mean = pt.compute_mean_direction(sites_geo)

ipmag.print_direction_mean(dir_mean)

ipmag.plot_net()
ipmag.plot_di(di_block=dir_block, color='blue', marker='o')
ipmag.plot_di_mean(dir_mean['dec'], dir_mean['inc'], dir_mean['alpha95'], 
                   color='red', marker='s')
ipmag.plt.title('Mean Site Directions (as measured at sites)')
ipmag.plt.show()

## Igneous-layering ("fold") and reversal tests

In [ ]:
# igneous layering test: the anorthosite layering acts like bedding
ipmag.plot_net()
ipmag.plot_di(sites_geo['dir_dec'].tolist(), sites_geo['dir_inc'].tolist(),
              color='red', marker='o', label='in-situ')
ipmag.plot_di(sites_tc['dir_dec'].tolist(), sites_tc['dir_inc'].tolist(),
              color='blue', marker='s', label='layering-corrected')
plt.legend(loc='center left', bbox_to_anchor=(1.05, 0.5))
plt.title('Rocky Mountain intrusions: in-situ vs. layering-corrected directions')
plt.show()

In [ ]:
# Find sites that have both geographic and tilt-corrected coordinates
sites_geolist = []
for site in sites_geo["site"]:
    if sites_tc["site"].str.contains(site, regex=False).any():
        sites_geolist.append(site)

decs_fold = []
incs_fold = []
dip_dirs = []
dips = []

for site in sites_geolist:
    dec_geo = float(sites_geo.loc[sites_geo["site"] == site, "dir_dec"].iloc[0])
    inc_geo = float(sites_geo.loc[sites_geo["site"] == site, "dir_inc"].iloc[0])
    dec_tilt = float(sites_tc.loc[sites_tc["site"] == site, "dir_dec"].iloc[0])
    inc_tilt = float(sites_tc.loc[sites_tc["site"] == site, "dir_inc"].iloc[0])

    dip_dir, dip = pmag.get_tilt(dec_geo, inc_geo, dec_tilt, inc_tilt)

    decs_fold.append(dec_geo)
    incs_fold.append(inc_geo)
    dip_dirs.append(dip_dir)
    dips.append(dip)

diddd = ipmag.make_diddd_array(decs_fold, incs_fold, dip_dirs, dips)

ipmag.bootstrap_fold_test(diddd)

The "fold test" definitively fails.

In [ ]:
# reversal test on the layering-corrected directions
try:
    ipmag.reversal_test_bootstrap(dec=sites_tc['dir_dec'].tolist(),
                                  inc=sites_tc['dir_inc'].tolist())
except Exception as e:
    print('reversal test:', e)

In [ ]:
ipmag.reversal_test_MM1990(dec=sites_tc['dir_dec'].tolist(), inc=sites_tc['dir_inc'].tolist())

The reversal test is indeterminate.

## R2: Deenen test

In [ ]:
pt.Deenen_test(pole_mean['n'], pole_mean['alpha95'])
pt.plot_Deenen_test(pole_mean)
plt.show()

## Paleosecular variation and VGP-shape diagnostics

The Fisher-quantile (fishqq) test assesses whether the site VGPs are drawn from a
Fisher distribution, and the shape-elongation/inclination (SVEI) test compares the
VGP-scatter elongation against the TK03.GAD paleosecular-variation model expectation
at the sampling paleolatitude.

In [ ]:
# All

fishqq_result = pt.fishqq_vgps(sites_geo, unify_polarity=True)
fishqq_result

In [ ]:
# Laramie Anorthosite

fishqq_result = pt.fishqq_vgps(subs[0], unify_polarity=True)
fishqq_result

In [ ]:
# Sherman Granite

fishqq_result = pt.fishqq_vgps(subs[1], unify_polarity=True)
fishqq_result

In [ ]:
# Electra Lake Gabbro

fishqq_result = pt.fishqq_vgps(subs[2], unify_polarity=True)
fishqq_result

In [ ]:
# All

_slat, _slon = float(sites_geo['lat'].mean()), float(sites_geo['lon'].mean())
try:
    svei_result = pt.svei_test_vgps(sites_geo, _slon, _slat, model='TK03_GAD', plot=True)
except TypeError:
    svei_result = pt.svei_test_vgps(sites_geo, _slon, _slat, model='TK03_GAD', plot=False)
    print('(SVEI elongation plot skipped: E below the TK03.GAD model minimum)')
print(f"paleolatitude = {svei_result['lat']:.1f} deg; elongation E = {svei_result['E']:.2f} "
      f"({'consistent' if svei_result['E_result'] else 'inconsistent'} with TK03.GAD)")

In [ ]:
# Laramie Anorthosite

_slat, _slon = float(subs[0]['lat'].mean()), float(subs[0]['lon'].mean())
try:
    svei_result = pt.svei_test_vgps(subs[0], _slon, _slat, model='TK03_GAD', plot=True)
except TypeError:
    svei_result = pt.svei_test_vgps(subs[0], _slon, _slat, model='TK03_GAD', plot=False)
    print('(SVEI elongation plot skipped: E below the TK03.GAD model minimum)')
print(f"paleolatitude = {svei_result['lat']:.1f} deg; elongation E = {svei_result['E']:.2f} "
      f"({'consistent' if svei_result['E_result'] else 'inconsistent'} with TK03.GAD)")

In [ ]:
# Sherman Granite

_slat, _slon = float(subs[1]['lat'].mean()), float(subs[1]['lon'].mean())
try:
    svei_result = pt.svei_test_vgps(subs[1], _slon, _slat, model='TK03_GAD', plot=True)
except TypeError:
    svei_result = pt.svei_test_vgps(subs[1], _slon, _slat, model='TK03_GAD', plot=False)
    print('(SVEI elongation plot skipped: E below the TK03.GAD model minimum)')
print(f"paleolatitude = {svei_result['lat']:.1f} deg; elongation E = {svei_result['E']:.2f} "
      f"({'consistent' if svei_result['E_result'] else 'inconsistent'} with TK03.GAD)")

In [ ]:
# Electra Lake Gabbro

_slat, _slon = float(subs[2]['lat'].mean()), float(subs[2]['lon'].mean())
try:
    svei_result = pt.svei_test_vgps(sub, _slon, _slat, model='TK03_GAD', plot=True)
except TypeError:
    svei_result = pt.svei_test_vgps(sub, _slon, _slat, model='TK03_GAD', plot=False)
    print('(SVEI elongation plot skipped: E below the TK03.GAD model minimum)')
print(f"paleolatitude = {svei_result['lat']:.1f} deg; elongation E = {svei_result['E']:.2f} "
      f"({'consistent' if svei_result['E_result'] else 'inconsistent'} with TK03.GAD)")

## The Rocky Mountain pole in the context of the Laurentia APWP

In [ ]:
# adopted prior-compilation mean pole
rocky_pole = {'inc': -11.9, 'dec': 217.4, 'alpha95': 9.7, 'n': 58}
Laurentia_poles = pt.get_Laurentia_poles()
ax = pt.plot_apwp_context(Laurentia_poles, pole_mean['inc'], pole_mean['dec'],
                          pole_mean['alpha95'], age_min=635, age_max=1800,
                          projection='orthographic',
                          central_longitude=pole_mean['dec'],
                          central_latitude=pole_mean['inc'])
plt.show()

## R7: comparison with younger Laurentia poles

In [ ]:
Laurentia_stricto_poles = pt.get_Laurentia_stricto_poles()
pt.plot_pole_overlap('MEAN Rocky Mountain intrusions', Laurentia_stricto_poles,
                     pt.Torsvik2012_Laurentia,
                     pole_plat=pole_mean['inc'], pole_plon=pole_mean['dec'],
                     pole_A95=pole_mean['alpha95'], pole_age=1430)

In [ ]:
dir_mean

## R-score summary (Meert et al., 2020)

| R | Criterion | Score | Justification |
|---|---|---|---|
| 1 | Age within ± 15 Ma | **0** | A combined pole spanning three intrusions of ca. 1415-1445 Ma; the ~30 Ma spread exceeds ± 15 Ma. |
| 2 | Techniques and statistical analysis | **1** | (a) The Laramie Anorthosite and Sherman Granite were demagnetized using only AF, while the Electra Lake Gabbro was demagnetized using both AF and thermal. (b) PCA used. (c) N = 448 samples from 62 sites, k = 10.1, A95 = 5.9ª barely passes Deenan et al. (2011) envelope (2.3º-6.1º), and B = 62 sites ≥ 8 sites. |
| 3 | Magnetic mineralogy characterized | **1** | Magnetite remanence characterized (Harlan et al., 1994, 1998). Experiments include IRM acquisition and demagnetization, ARM demagnetization, Curie temperature determinations, susceptibilty vs. temperature, and microscopy + EDS. |
| 4 | Field tests constrain age of magnetization | **0** | There are no decisive field tests. |
| 5 | Structural control / tectonic coherence | **0** | A combined pole across three separate intrusions and structures. No correction for regional tilting. |
| 6 | Presence of reversals | **0** | The directions do not pass a reversal test. |
| 7 | No resemblance to younger poles | **1** | Distinct from younger Laurentia poles. |
| | Total | **3/7** | Grade B |

## Nordic workshop summary

The Rocky Mountain intrusions pole is reported as the prior compilation's mean of
the three ~1.4 Ga study poles (Laramie Anorthosite + Sherman Granite, Harlan et
al. 1994; Electra Lake gabbro, Harlan et al. 1998): −11.9°N/217.4°E, A95 9.7°,
N=58, mean direction 41.1°/−46.6°. The pole was recalculated using site VGPs: −10.3°N/217.9°E, A95 5.9°,
N=62, mean direction 41.4°/−48.3°. We recommend retaining the 'B' rating. Future workshops may also consider splitting this mean into multiple poles, given that the three intrusions do not share a common mean.

## Pole comparison

### Mean Rocky Mountain intrusions pole

This notebook reports the prior-compilation mean (the Buchan/Luleå working-group mean of the three ~1.4 Ga study poles), which is the value exported here. There is no single published three-study mean direction equal to the compilation value; the closest published *combined* pole is the Harlan et al. (1994) combined Laramie Anorthosite Complex + Sherman Granite VGP-mean (Electra Lake gabbro added separately in Harlan et al., 1998). That published combined pole is given below for reference; rows it does not supply are marked "—".

| | This study (site VGPs) | Harlan et al. (1994) combined LAC+SG | Previous Nordic compilation |
|---|---|---|---|
| Component | combined ChRM (magnetite), 3 intrusions | combined LAC + Sherman Granite | MEAN Rocky Mountain intrusions |
| N (sites) | 62 | 37 (VGPs) | 58 |
| N (specimens) | 448 | — | 434 |
| Dec (°) | 41.4 | — | 41.1 |
| Inc (°) | −48.3 | — | −46.6 |
| k (direction) | 11.5 | — | 1000 |
| α95 (direction, °) | 5.6 | — | 0.1 |
| Pole lat (°N) | −10.3 | −6.7 | −11.9 |
| Pole lon (°E) | 217.9 | 215.0 | 217.4 |
| A95 | 5.9 | 3.5 | 9.7 |
| pole (°) | — | — | 9.7 |
| dp / dm (°) | — | — | 9.7 / 9.7 |


In [ ]:
rocky_summary = pt.make_nordic_summary(
    terrane='Laurentia',
    rockname='Mean Rocky Mountain intrusions',
    sites=sites_geo,
    dir_mean=dir_mean,
    pole_mean=pole_mean,
    study_lon=study_lon,
    study_lat=study_lat,
    component_comment='Characteristic remanent magnetization (magnetite); combined mean of three ca. 1.4 Ga Colorado-Wyoming intrusions',
    tests='',
    gpmdb_number='7493:7494:8342',
    percent_reversed=0,
    demag_code=4,
    R1=0, R2=1, R3=1, R4='', R5=0, R6=0, R7=1, Grade='B',
    nominal_age=1430, lomagage=1415, himagage=1445,
    REF_method='Combined pole of three ~1.4 Ga Colorado-Wyoming anorogenic intrusions: Laramie Anorthosite Complex and Sherman Granite (Harlan et al., 1994) and Electra Lake gabbro (Harlan et al., 1998); ages span ca. 1415-1445 Ma.',
    POLE_AUTHORS='Harlan, S. S., Snee, L. W., Geissman, J. W., & Brearley, A. J.; Harlan, S. S., & Geissman, J. W.',
    YEAR='1994; 1998',
    JOURNAL='Journal of Geophysical Research: Solid Earth; ',
    VOLUME='99; 103',
    VPAGES='17997-18020; 15497-15507',
    TITLE='Paleomagnetism of the Middle Proterozoic Laramie anorthosite complex and Sherman Granite, southern Laramie Range, Wyoming and Colorado; Paleomagnetism of the Middle Proterozoic Electra Lake Gabbro, Needle Mountains, southwestern Colorado',
    COMMENT='Recalculated by Laurie Zielinski as documented in Swanson-Hysell et al. (2026) notebook using site VGPs'
)
pt.save_nordic_summary(rocky_summary, '1430_Rocky_Mountain_intrusions')
rocky_summary